In [15]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [16]:
PROJECT_ROOT = Path().resolve()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

In [17]:
from src.labeling import add_returns, add_volatility, add_volatility_regime
from src.features import (
    add_target_direction,
    add_lagged_returns,
    add_moving_average_features,
    add_volatility_features
)
df = pd.read_csv("../data/raw/nifty50.csv", parse_dates=["Date"])
df.set_index("Date", inplace=True)

df = add_returns(df)
df = add_volatility(df)
df = add_volatility_regime(df)

df = add_target_direction(df)
df = add_lagged_returns(df)
df = add_moving_average_features(df)
df = add_volatility_features(df)

df_feat = df.dropna().copy()
df_feat.head()

,Open,High,Low,Close,Volume,return,volatility,vol_regime,target,ret_lag_1,ret_lag_5,ret_lag_10,ma_ratio_5,ma_ratio_10,ma_ratio_20,vol_lag_1
Date,,,,,,,,,,,,,,,,
2007-10-17,5658.899902,5658.899902,5107.299805,5559.299805,0,-0.019186,0.312725,High,0,-0.000414,0.021437,0.027984,0.998049,1.027613,1.073952,0.292689
2007-10-18,5551.100098,5736.799805,5269.649902,5351.000000,0,-0.037469,0.333930,High,0,-0.019186,0.015327,-0.000413,0.966687,0.986514,1.027572,0.312725
2007-10-19,5360.350098,5390.850098,5101.750000,5215.299805,0,-0.025360,0.352405,High,0,-0.037469,-0.017485,-0.004377,0.949478,0.960974,0.997035,0.333930
2007-10-22,5202.750000,5247.399902,5070.899902,5184.000000,0,-0.006002,0.350371,High,1,-0.025360,0.044609,-0.019428,0.960795,0.953470,0.987780,0.352405
2007-10-23,5185.299805,5488.500000,5176.850098,5473.700195,0,0.055884,0.393280,High,1,-0.006002,-0.000414,0.047619,1.021849,1.004048,1.037627,0.350371


# Added vol_regime

In [18]:
model_cols = [
    "target",
    "ret_lag_1","ret_lag_5","ret_lag_10",
    "ma_ratio_5","ma_ratio_10","ma_ratio_20",
    "volatility","vol_lag_1",
    "vol_regime"
]

df_model = df_feat[model_cols].copy()

In [19]:
split_idx = int(len(df_model) * 0.8)

train = df_model.iloc[:split_idx]
test  = df_model.iloc[split_idx:]

In [20]:
X_train = train.drop(columns=["target","vol_regime"])
y_train = train["target"]

X_test = test.drop(columns=["target","vol_regime"])
y_test = test["target"]

In [21]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=X_test.columns,
    index=X_test.index
)

In [22]:
baseline_model = LogisticRegression(max_iter=1000)
baseline_model.fit(X_train_scaled, y_train)

pred_base = baseline_model.predict(X_test_scaled)

baseline_acc = accuracy_score(y_test, pred_base)
print("Baseline accuracy:", baseline_acc)

Baseline accuracy: 0.5477777777777778


# One Hot Encode Regime
1. **converts categorical regime -> numeric columns**
2. **align ensures same columns in train/test**

In [23]:
train_reg = pd.get_dummies(train["vol_regime"], prefix="regime")
test_reg  = pd.get_dummies(test["vol_regime"],  prefix="regime")

train_reg, test_reg = train_reg.align(test_reg, axis=1, fill_value=0)

In [24]:
X_train_reg = pd.concat([X_train_scaled, train_reg], axis=1)
X_test_reg  = pd.concat([X_test_scaled,  test_reg],  axis=1)
#Now model sees both numeric features and regime state.

# Train Regime feature model

In [25]:
regime_feature_model = LogisticRegression(max_iter=1000)
regime_feature_model.fit(X_train_reg, y_train)

pred_reg = regime_feature_model.predict(X_test_reg)

regime_feature_acc = accuracy_score(y_test, pred_reg)

print("Regime-feature accuracy:", regime_feature_acc)

Regime-feature accuracy: 0.5455555555555556


In [26]:
print("Baseline accuracy:", baseline_acc)
print("Regime-feature accuracy:", regime_feature_acc)

Baseline accuracy: 0.5477777777777778
Regime-feature accuracy: 0.5455555555555556


In [27]:
print("Regime-specific model accuracies:")

for regime in ["Low","Medium","High"]:
    
    train_r = train[train["vol_regime"] == regime]
    test_r  = test[test["vol_regime"] == regime]
    
    if len(test_r) == 0:
        continue
    
    X_tr = X_train_scaled.loc[train_r.index]
    y_tr = y_train.loc[train_r.index]
    
    X_te = X_test_scaled.loc[test_r.index]
    y_te = y_test.loc[test_r.index]
    
    model_r = LogisticRegression(max_iter=1000)
    model_r.fit(X_tr, y_tr)
    
    pred_r = model_r.predict(X_te)
    
    acc_r = accuracy_score(y_te, pred_r)
    
    print(regime, ":", acc_r, " (n=", len(test_r), ")")

Regime-specific model accuracies:
Low : 0.5555555555555556  (n= 576 )
Medium : 0.5311203319502075  (n= 241 )
High : 0.5421686746987951  (n= 83 )
